# 🧱 Notebook 5: Stacking All the Patterns Together

The previous four notebooks each showed **one** resilience pattern in isolation. Real production clients layer them together. This capstone wires them into **one** request pipeline you can run end-to-end.

**The failure modes we're defending against — all at once:**

| Failure | Pattern that handles it |
|---|---|
| A single call hangs forever | **Timeout** |
| A transient network blip | **Retry with jitter** |
| The downstream is *sustainedly* dead — retrying is pointless | **Circuit breaker** |
| Too many concurrent calls saturate threads / connections | **Bulkhead** (bounded semaphore) |
| Everything above failed — but we still need to show the user *something* | **Fallback** (cached / default) |

**The scenario:** 20 concurrent users request the product page. The recommendations service is **broken and slow** for the first ~2 seconds, then recovers. We'll compare:

- 🟥 **Naive** — direct call, no resilience.
- 🟩 **Stacked** — all five patterns together.

**What "good" looks like:** the stacked version should finish fast (seconds, not minutes), serve *every* user a useful page (fresh when possible, cached/default otherwise), and stop hammering the downstream once the breaker trips.


## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 The misbehaving downstream

Our fake `recommendations_api` takes **0.4 s** on every call and raises `IOError` while `broken=True`. Enough to be annoying, short enough that the notebook finishes quickly.

We also track a call counter so we can *see* the breaker saving trips to the downstream.


In [ ]:
import time, threading, random, itertools
from concurrent.futures import ThreadPoolExecutor

# Shared mutable "world": toggle to recover the downstream mid-demo.
broken = True
downstream_call_count = 0
_counter_lock = threading.Lock()

def recommendations_api(user: str) -> list[str]:
    """Fake downstream. Always takes 0.4s. Fails while `broken` is True."""
    global downstream_call_count
    with _counter_lock:
        downstream_call_count += 1
    time.sleep(0.15)                        # simulates network + server work
    if broken:
        raise IOError('recs service 503')
    return [f'{user}-pick-1', f'{user}-pick-2']


## 🟥 BASELINE: naive direct call (no resilience)

20 concurrent users, every request goes straight through. While the service is broken, every user sees an error *and* we waste 20 calls on a service we already know is down.


In [ ]:
def naive_product_page(user):
    return {'user': user, 'recs': recommendations_api(user)}

def run_naive(n=20):
    global downstream_call_count
    downstream_call_count = 0
    results = {'ok': 0, 'error': 0}
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n) as pool:
        futs = [pool.submit(naive_product_page, f'user{i}') for i in range(n)]
        for f in futs:
            try:
                f.result()
                results['ok'] += 1
            except Exception:
                results['error'] += 1
    elapsed = time.perf_counter() - t0
    print(f'[naive]  {results}  downstream calls: {downstream_call_count}  elapsed: {elapsed:.2f}s')

broken = True
run_naive()


20 users → 20 errors, 20 calls hammered the already-dying service, and *every* user saw a broken page. Now let's stack the patterns.


## 🟩 STACKED: the resilient pipeline

We build each component **minimally** — all stdlib, no magic — then compose them. Each piece is a refinement of what you saw in notebooks 1-4.

### Piece 1 — thread-safe Circuit Breaker

Notebook 2's breaker wasn't thread-safe; with 20 concurrent callers we need a `Lock` so two threads don't both see `fails == threshold - 1` and both trip.


In [ ]:
class CircuitBreaker:
    def __init__(self, fail_threshold=4, cooldown=1.0, name='breaker'):
        self.name = name
        self.fail_threshold = fail_threshold
        self.cooldown = cooldown
        self.fails = 0
        self.opened_at = 0.0
        self.state = 'CLOSED'
        self._lock = threading.Lock()      # ⬅ new: serialize state changes
        self.fast_fail_count = 0           # observability

    def _before(self):
        with self._lock:
            if self.state == 'OPEN':
                if time.monotonic() - self.opened_at >= self.cooldown:
                    self.state = 'HALF_OPEN'
                    return 'probe'         # one thread gets to probe
                self.fast_fail_count += 1
                return 'fast_fail'
            return 'pass'

    def _after(self, success: bool):
        with self._lock:
            if success:
                self.state, self.fails = 'CLOSED', 0
            else:
                self.fails += 1
                if self.state == 'HALF_OPEN' or self.fails >= self.fail_threshold:
                    if self.state != 'OPEN':
                        print(f'  ⚡ breaker[{self.name}] OPENED')
                    self.state = 'OPEN'
                    self.opened_at = time.monotonic()

    def call(self, fn, *args, **kwargs):
        decision = self._before()
        if decision == 'fast_fail':
            raise RuntimeError(f'breaker[{self.name}] OPEN — fast-failing')
        try:
            result = fn(*args, **kwargs)
        except Exception:
            self._after(success=False)
            raise
        self._after(success=True)
        return result


### Piece 2 — per-attempt timeout

Same helper as notebook 4: run the call in a daemon thread; if it doesn't finish in time, raise. ⚠️ **Caveat (worth teaching):** Python can't *cancel* arbitrary blocking code, so the worker thread keeps running in the background until the sleep/IO completes. In production, use the native `timeout=` argument of your HTTP/DB client — it actually aborts the socket.


In [ ]:
class CallTimeout(Exception): pass

def with_timeout(fn, timeout, *args, **kwargs):
    result, error, done = [], [], threading.Event()

    def worker():
        try: result.append(fn(*args, **kwargs))
        except Exception as e: error.append(e)
        finally: done.set()

    threading.Thread(target=worker, daemon=True).start()
    if not done.wait(timeout):
        raise CallTimeout(f'exceeded {timeout}s')
    if error: raise error[0]
    return result[0]


### Piece 3 — retry with full jitter

Exactly the AWS recipe from notebook 1. **Put retry *inside* the breaker**: the breaker counts one *overall* failure, not one per retry attempt.


In [ ]:
def retry_with_jitter(fn, attempts=3, base=0.05, cap=0.3, *args, **kwargs):
    last_exc = None
    for i in range(attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_exc = e
            wait = random.uniform(0, min(cap, base * 2**i))
            time.sleep(wait)
    raise last_exc


### Piece 4 — bulkhead as a bounded semaphore

`ThreadPoolExecutor` *alone* isn't a fast-fail bulkhead — it has an **unbounded queue**, so under overload, new callers just queue forever. A real bulkhead **rejects** work once saturated so callers can immediately use the fallback instead of waiting.

Here: `Semaphore(max_concurrency)` with `acquire(blocking=False)`. No slot → instant `BulkheadFull` → fallback.


In [ ]:
class BulkheadFull(Exception): pass

class Bulkhead:
    def __init__(self, max_concurrency=6, name='bulkhead'):
        self.sem = threading.Semaphore(max_concurrency)
        self.name = name
        self.reject_count = 0
        self._lock = threading.Lock()

    def run(self, fn, *args, **kwargs):
        if not self.sem.acquire(blocking=False):   # fast-fail when full
            with self._lock: self.reject_count += 1
            raise BulkheadFull(f'bulkhead[{self.name}] full')
        try:
            return fn(*args, **kwargs)
        finally:
            self.sem.release()


### Piece 5 — fallback (last-good cache → generic default)

Same two-tier fallback as notebook 4. This is what the user actually sees when everything else raises.


In [ ]:
POPULAR = ['socks', 't-shirt', 'mug']
_recs_cache: dict[str, list[str]] = {}

def get_fallback(user):
    if user in _recs_cache:
        return {'source': 'stale_cache', 'recs': _recs_cache[user]}
    return {'source': 'popular_default', 'recs': POPULAR}


### Putting it all together

Nesting order (outermost → innermost) — **read it top to bottom like the request flows down**:

```
fallback        ← last-resort safety net (catches anything below)
  bulkhead      ← admission control: reject if too many in flight
    breaker     ← short-circuit if downstream has been failing
      retry     ← survive transient blips (INSIDE the breaker!)
        timeout ← bound each individual attempt
          recommendations_api(user)
```


In [ ]:
breaker  = CircuitBreaker(fail_threshold=4, cooldown=1.0, name='recs')
bulkhead = Bulkhead(max_concurrency=6, name='recs')

stats = {'fresh': 0, 'stale_cache': 0, 'popular_default': 0, 'bulkhead_rejected': 0}
_stats_lock = threading.Lock()
def bump(k):
    with _stats_lock: stats[k] = stats.get(k, 0) + 1

def get_recommendations(user):
    """Single protected call attempt: breaker → retry → timeout → api."""
    def one_attempt():
        return with_timeout(recommendations_api, 0.5, user)
    return breaker.call(lambda: retry_with_jitter(one_attempt, attempts=3))

def product_page(user):
    try:
        fresh = bulkhead.run(get_recommendations, user)
        _recs_cache[user] = fresh          # refresh last-good cache on success
        bump('fresh')
        return {'user': user, 'source': 'fresh', 'recs': fresh}
    except BulkheadFull:
        bump('bulkhead_rejected')
        fb = get_fallback(user); bump(fb['source'])
        return {'user': user, **fb}
    except Exception:
        # Breaker OPEN, retries exhausted, or timeout — all caught here.
        fb = get_fallback(user); bump(fb['source'])
        return {'user': user, **fb}


## 💥 Run the stacked version

20 concurrent users hitting the broken-then-recovering service. Watch:
- the breaker trips after ~4 failures and starts **fast-failing**,
- most users get a **fallback** page (default, since cache is empty),
- total downstream calls is **small** — the breaker stops the bleeding.


In [ ]:
def run_stacked(n=20, label='stacked'):
    global downstream_call_count
    downstream_call_count = 0
    for k in list(stats): stats[k] = 0
    breaker.fast_fail_count = 0
    bulkhead.reject_count = 0

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n) as pool:
        pages = list(pool.map(product_page, [f'user{i}' for i in range(n)]))
    elapsed = time.perf_counter() - t0

    print(f'[{label}] elapsed: {elapsed:.2f}s  downstream calls: {downstream_call_count}')
    print(f'          breaker fast-fails: {breaker.fast_fail_count}  bulkhead rejects: {bulkhead.reject_count}')
    print(f'          result mix: {stats}  breaker state: {breaker.state}')
    return pages

broken = True
run_stacked(label='broken  ');


Compare to the naive baseline: same 20 users, but now everyone got a **page** (not a 500), and the downstream was spared most of the load.


## 🔁 And when the downstream recovers…

We flip `broken=False`, wait out the breaker cooldown, and run again. The breaker probes, succeeds, closes, and everyone gets **fresh** recommendations.


In [ ]:
broken = False
time.sleep(1.1)                 # wait out breaker cooldown
run_stacked(label='recovered');


Notice:

- **Downstream calls stayed small (~6)** — the bulkhead capped concurrency, so even 20 eager users can't stampede a just-recovered service. That's a feature, not a bug: resilience chooses *"14 users see a popular-items fallback for one second"* over *"downstream gets a thundering herd and dies again"*.
- **Breaker state: CLOSED** — the first HALF_OPEN probe succeeded, so the breaker closed and subsequent calls went through normally.
- Run the cell again and you'll see the cache kick in: previously-successful users now get `stale_cache` if a new failure happens.


## 🧪 Try it yourself

Tweak these and re-run the "broken" scenario to build intuition:

- **`fail_threshold`** — lower (e.g. 2) trips faster (less downstream load) but is twitchier on noisy deps.
- **`cooldown`** — longer protects a sick downstream more; too long hurts recovery.
- **`max_concurrency`** on the bulkhead — lower rejects more users straight to fallback (faster page, less freshness).
- **`attempts`** on retry — more retries smooth over transient blips, but also amplify load (retry storms).
- **`timeout`** — too low → false positives (healthy slow calls treated as failed); too high → users wait.

There's no universally right answer — these are **knobs** you tune with real traffic data and load tests.


## 🌍 Things a real production stack would add

This capstone is intentionally minimal. In production you'd also see:

- **Native client timeouts** (`requests.get(url, timeout=...)`, `httpx`, DB driver timeouts) instead of the thread-based helper — those actually abort the socket.
- **Idempotency keys** on every retried write so a retry storm doesn't double-charge a card.
- **Sliding-window breakers** (rate-based, not consecutive-count) — e.g. trip if >50% of the last 20 calls failed.
- **Metrics + alerts** on breaker state transitions (`CLOSED → OPEN` = page the on-call).
- **Distributed tracing** (OpenTelemetry) so you can see where in the stack time was spent.
- **Load shedding at the edge** — return 503 with `Retry-After` when overloaded, instead of queueing.
- **Feature flags / kill switches** (notebook 4) to disable expensive features during an incident without a deploy.
- **Chaos engineering** — deliberately injecting failures (Netflix's Chaos Monkey) to prove your resilience actually works.

Batteries-included libraries: [`tenacity`](https://tenacity.readthedocs.io/) (retries), [`pybreaker`](https://github.com/danielfm/pybreaker) (circuit breaker), [resilience4j](https://resilience4j.readme.io/) (JVM, the reference implementation of all these patterns).


## 🧠 Takeaway

Resilience patterns are **not alternatives** — they're layers, and each one catches a failure mode the others can't:

- **Timeout** bounds a single call.
- **Retry** handles transient blips.
- **Circuit breaker** handles sustained outages.
- **Bulkhead** bounds blast radius when something misbehaves.
- **Fallback** keeps the user's experience from cratering when all else fails.

> "Your system will fail. Resilience is what determines whether users notice."
